# Project Milestone 4

## Connecting to an API and cleaning data

In [1]:
#import libraries
import requests
import pandas as pd

In [4]:
#api_url
url = 'http://universities.hipolabs.com'
search_str = '/search?all'
request_url = url+search_str

try:
    # get response and raise status
    response = requests.get(request_url)
    response.raise_for_status()

    # get data
    data = response.json()

    # save as df
    df = pd.DataFrame(data)
except Exception as e:
    print('An error occurred:',e)

An error occurred: ('Connection broken: IncompleteRead(104929 bytes read, 1829099 more expected)', IncompleteRead(104929 bytes read, 1829099 more expected))


In [5]:
df = pd.read_json('world_universities_and_domains.json')

In [6]:
df.head()

,name,domains,web_pages,country,alpha_two_code,state-province
0,Engineering Institute of Technology,[student.eit.edu.au],[https://www.eit.edu.au/],Australia,AU,None
1,Universitas Nusa Putra,[nusaputra.ac.id],[https://nusaputra.ac.id],Indonesia,ID,None
2,University of Kyrenia,"[std.kyrenia.edu.tr, kyrenia.edu.tr]",[https://kyrenia.edu.tr],Turkey,TR,None
3,Regent University College of Science and Techn...,[regent.edu.gh],[https://regent.edu.gh],Ghana,GH,None
4,Wroclaw Akademia Biznesu,"[student.wab.edu.pl, wab.edu.pl]",[https://wab.edu.pl],Poland,PL,None


### Step 1. Flatten list-type columns

Flatten list type fields ('domains' and 'web_pages'). 

'domains' may contain multiple values. Joining them into a single string allows for easier comparison.

'web_pages' often has one URL. Taking the first instance ensures a consisten format across the dataset.

These changes simplify the structure and make the dataset more suitable for integration with other sources.

In [7]:
#flatten columns
df['domains'] = df['domains'].apply(lambda x: ', '.join(x))
df['web_pages'] = df['web_pages'].apply(lambda x: x[0])


In [8]:
df[['domains','web_pages']].head()

,domains,web_pages
0,student.eit.edu.au,https://www.eit.edu.au/
1,nusaputra.ac.id,https://nusaputra.ac.id
2,"std.kyrenia.edu.tr, kyrenia.edu.tr",https://kyrenia.edu.tr
3,regent.edu.gh,https://regent.edu.gh
4,"student.wab.edu.pl, wab.edu.pl",https://wab.edu.pl


### Step 2. Standardize country names

Convert all country names to lowercase and strip any leading/trailing whitespaces.

By doing so, it ensures consistent formatting across other sources and prevents merge failures due to minor mismatches like "United States" vs "united states"

In [9]:
# strip and lowercase country names
df['country'] = df['country'].str.strip().str.lower()

In [10]:
df['country'].head()

0    australia
1    indonesia
2       turkey
3        ghana
4       poland
Name: country, dtype: object

### Step 3: Drop unnecessary columns

Drop 'state-province' because it is mostly nulls and the project is focused on country-level data.

Drop 'alpha_two_code' because it is redundant with 'country' and will not be used to merge with other data sources.

Drop 'web_pages' because it typically duplicates what's already in 'domains'. It rarely provides addtional unique information.

In [11]:
# drop columns
df.drop(columns=['state-province','alpha_two_code','web_pages'], inplace=True)

In [12]:
df.head()

,name,domains,country
0,Engineering Institute of Technology,student.eit.edu.au,australia
1,Universitas Nusa Putra,nusaputra.ac.id,indonesia
2,University of Kyrenia,"std.kyrenia.edu.tr, kyrenia.edu.tr",turkey
3,Regent University College of Science and Techn...,regent.edu.gh,ghana
4,Wroclaw Akademia Biznesu,"student.wab.edu.pl, wab.edu.pl",poland


### Step 4: Extract Top-Level Domain (TLD)

Extract Top-Level Domain (TLD) from 'domains'. It often indicate the type of univerisity or national structure (.edu = U.S. education, .ac = U.K. academic). It helps analyze differences in educational infrastructure across countries.

In [13]:
df['tld'] = df['domains'].apply(lambda x: x.split('.')[-1])

In [14]:
df['tld'].head()

0    au
1    id
2    tr
3    gh
4    pl
Name: tld, dtype: object

### Step 5: Aggregate by country

Group the dataset by country and create a new dataframe that contains total number of universities and most frequently used TLD in that country.

This tranforms institution-level data into country-level summary to align with the project goal.

It also allows integration with literacy and socioeconomic data that are already in country-level.

In [15]:
# create a new df with aggregated data by country
country_df = df.groupby('country').agg(
    university_count = ('name', 'count'),
    tld_mode = ('tld', lambda x: x.mode().iloc[0] if not x.mode().empty else None)
).reset_index()

In [16]:
country_df.sort_values(by='university_count',ascending=False).head()

,country,university_count,tld_mode
192,united states,2348,edu
91,japan,572,jp
83,india,473,in
37,china,397,cn
67,germany,318,de


### Final Output

In [17]:
country_df.head(25)

,country,university_count,tld_mode
0,afghanistan,40,af
1,albania,16,al
2,algeria,29,dz
3,andorra,1,ad
4,angola,8,ao
5,antigua and barbuda,2,ag
6,argentina,86,ar
7,armenia,12,am
8,australia,57,au
9,austria,46,at


In [18]:
country_df.to_csv("api.csv",index=False)